Homework 5.
Amanda Kim.


In [1]:
# Homework 5
# Import packages

import pandas as pd
import numpy as np
import requests
import plotly.express as px
import plotly.graph_objects as go

Part 1: Read the GDP tables from Wikipedia

In [2]:
# Read GDP tables from Wikipedia

from io import StringIO

url = "https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467"

# Request the webpage using a browser-like header
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers) ## requests.get retrieves the webpage before passing it to pandas
response.raise_for_status()

# Extract tables from the webpage
tables = pd.read_html(StringIO(response.text))

print(len(tables))

7


In [3]:
# Examine the tables extracted from Wikipedia

for i, table in enumerate(tables):
    print("Table", i)
    print("Dimensions:", table.shape)
    print("Columns:", table.columns.tolist())
    print()

Table 0
Dimensions: (1, 1)
Columns: [0]

Table 1
Dimensions: (1, 3)
Columns: [0, 1, 2]

Table 2
Dimensions: (214, 8)
Columns: [('Country/Territory', 'Country/Territory'), ('UN region', 'UN region'), ('IMF[1][13]', 'Forecast'), ('IMF[1][13]', 'Year'), ('World Bank[14]', 'Estimate'), ('World Bank[14]', 'Year'), ('United Nations[15]', 'Estimate'), ('United Nations[15]', 'Year')]

Table 3
Dimensions: (8, 2)
Columns: ['vteLists of countries by financial rankings', 'vteLists of countries by financial rankings.1']

Table 4
Dimensions: (9, 2)
Columns: ['vteLists of countries by GDP rankings', 'vteLists of countries by GDP rankings.1']

Table 5
Dimensions: (13, 2)
Columns: ['vteEconomic classification of countries', 'vteEconomic classification of countries.1']

Table 6
Dimensions: (2, 2)
Columns: [0, 1]



Output: 214 rows representing countries, terrirotires, and potntially aggregate entries.
8 columns containing GDP estimates from the IMF, world bank, and UN.
It also has a UN region column that we can use to group countries geographically.

In [4]:
# Select the GDP table
gdp = tables[2]

# Keep country, region, and IMF GDP columns
gdp = gdp.iloc[:, [0, 1, 2, 3]]

# Rename columns
gdp.columns = ["country", "region", "gdp", "year"]

# View the first 10 rows
gdp.head(10)

,country,region,gdp,year
0,World,—,104476432,2023
1,United States,Americas,26949643,2023
2,China,Asia,17700899,[n 1]2023
3,Germany,Europe,4429838,2023
4,Japan,Asia,4230862,2023
5,India,Asia,3732224,2023
6,United Kingdom,Europe,3332059,2023
7,France,Europe,3049016,2023
8,Italy,Europe,2186082,2023
9,Brazil,Americas,2126809,2023


In [5]:
# Remove the World total
gdp = gdp[gdp["country"] != "World"].copy()

# Convert GDP values to numbers
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")

# Remove rows with missing GDP or region information
gdp = gdp.dropna(subset=["gdp", "region"])

# Check the cleaned data
print(gdp.shape)
print(gdp["region"].value_counts())

(191, 4)
region
Africa      53
Asia        46
Europe      42
Americas    36
Oceania     14
Name: count, dtype: int64


In [6]:
# Create an interactive stacked bar chart of GDP by region

fig = px.bar(
    gdp,
    x="region",
    y="gdp",
    color="country",
    title="GDP by Country and Region (IMF, 2023)",
    labels={
        "region": "Region",
        "gdp": "GDP (millions of US dollars)",
        "country": "Country"
    }
)

# Stack the bars
fig.update_layout(barmode="stack")

# Display the interactive chart
fig.show()

In [7]:
# Save the interactive stacked bar chart as an HTML file

fig.write_html("stacked_bar.html")

Part 2: Creating a Sankey diagram using MRICloud data

In [8]:
# Load MRICloud hierarchy data

url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"

multilevel_lookup = pd.read_csv(url, sep="\t").drop(["Level5"], axis=1)

multilevel_lookup = multilevel_lookup.rename(columns={
    "modify": "roi",
    "modify.1": "level4",
    "modify.2": "level3",
    "modify.3": "level2",
    "modify.4": "level1"
})

multilevel_lookup = multilevel_lookup[
    ["roi", "level4", "level3", "level2", "level1"]
]

# View the first five rows
multilevel_lookup.head()

,roi,level4,level3,level2,level1
0,SFG_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
1,SFG_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
2,SFG_PFC_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
3,SFG_PFC_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
4,SFG_pole_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L


In [9]:
# Load MRICloud data for subject 127

id = 127

subjectData = pd.read_csv(
    "https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv"
)

# Select Type = 1, Level = 5, and subject 127
subjectData = subjectData.loc[
    (subjectData.type == 1) &
    (subjectData.level == 5) &
    (subjectData.id == id)
]

# Keep only the region and volume columns
subjectData = subjectData[["roi", "volume"]]

# Merge with the anatomical hierarchy
subjectData = pd.merge(subjectData, multilevel_lookup, on="roi")

# Add intracranial volume as the starting level
subjectData = subjectData.assign(icv="ICV")

# Calculate each region's proportion of total volume
subjectData = subjectData.assign(
    comp=subjectData.volume / np.sum(subjectData.volume)
)

# View the first five rows
subjectData.head()

,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


In [10]:
# Calculate volume for each Level 1 brain region

level1_data = (
    subjectData.groupby(["icv", "level1"])["volume"]
    .sum()
    .reset_index()
)

# View the results
level1_data.head(10)

,icv,level1,volume
0,ICV,CSF,109787
1,ICV,Diencephalon_L,11817
2,ICV,Diencephalon_R,11560
3,ICV,Mesencephalon,10271
4,ICV,Metencephalon,159411
5,ICV,Myelencephalon,4975
6,ICV,Telencephalon_L,531153
7,ICV,Telencephalon_R,543444


In [11]:
# Calculate volume for each Level 1 to Level 2 connection

level2_data = (
    subjectData.groupby(["level1", "level2"])["volume"]
    .sum()
    .reset_index()
)

# View the first 10 rows
level2_data.head(10)

,level1,level2,volume
0,CSF,Sulcus_L,33975
1,CSF,Sulcus_R,30018
2,CSF,Ventricle,44979
3,CSF,Ventricle,815
4,Diencephalon_L,BasalForebrain_L,5475
5,Diencephalon_L,Thalamus_L,6342
6,Diencephalon_R,BasalForebrain_R,5188
7,Diencephalon_R,Thalamus_R,6372
8,Mesencephalon,Mesencephalon_L,4945
9,Mesencephalon,Mesencephalon_R,5326


In [12]:
# Calculate volume for each Level 2 to Level 3 connection

level3_data = (
    subjectData.groupby(["level1", "level2", "level3"])["volume"]
    .sum()
    .reset_index()
)

# View the first 10 rows
level3_data.head(10)

,level1,level2,level3,volume
0,CSF,Sulcus_L,CentralSul_L,2159
1,CSF,Sulcus_L,CinguSul_L,1173
2,CSF,Sulcus_L,FrontSul_L,15004
3,CSF,Sulcus_L,OcciptSul_L,1934
4,CSF,Sulcus_L,ParietSul_L,8739
5,CSF,Sulcus_L,SylvianFissureExt_L,3827
6,CSF,Sulcus_L,TempSul_L,1139
7,CSF,Sulcus_R,CentralSul_R,1750
8,CSF,Sulcus_R,CinguSul_R,1419
9,CSF,Sulcus_R,FrontSul_R,11232


In [13]:
# Calculate volume for each Level 3 to Level 4 connection

level4_data = (
    subjectData.groupby(["level1", "level2", "level3", "level4"])["volume"]
    .sum()
    .reset_index()
)

# View the first 10 rows
level4_data.head(10)

,level1,level2,level3,level4,volume
0,CSF,Sulcus_L,CentralSul_L,CentralSul_L,2159
1,CSF,Sulcus_L,CinguSul_L,CinguSul_L,1173
2,CSF,Sulcus_L,FrontSul_L,FrontSul_L,15004
3,CSF,Sulcus_L,OcciptSul_L,OcciptSul_L,1934
4,CSF,Sulcus_L,ParietSul_L,ParietSul_L,8739
5,CSF,Sulcus_L,SylvianFissureExt_L,SylFrontSul_L,1589
6,CSF,Sulcus_L,SylvianFissureExt_L,SylParieSul_L,640
7,CSF,Sulcus_L,SylvianFissureExt_L,SylTempSul_L,1598
8,CSF,Sulcus_L,TempSul_L,TempSul_L,1139
9,CSF,Sulcus_R,CentralSul_R,CentralSul_R,1750


In [14]:
# Calculate volume for each Level 4 to ROI connection

roi_data = (
    subjectData.groupby(
        ["level1", "level2", "level3", "level4", "roi"]
    )["volume"]
    .sum()
    .reset_index()
)

# View the first 10 rows
roi_data.head(10)

,level1,level2,level3,level4,roi,volume
0,CSF,Sulcus_L,CentralSul_L,CentralSul_L,CentralSul_L,2159
1,CSF,Sulcus_L,CinguSul_L,CinguSul_L,CinguSul_L,1173
2,CSF,Sulcus_L,FrontSul_L,FrontSul_L,FrontSul_L,15004
3,CSF,Sulcus_L,OcciptSul_L,OcciptSul_L,OcciptSul_L,1934
4,CSF,Sulcus_L,ParietSul_L,ParietSul_L,ParietSul_L,8739
5,CSF,Sulcus_L,SylvianFissureExt_L,SylFrontSul_L,SylFrontSul_L,1589
6,CSF,Sulcus_L,SylvianFissureExt_L,SylParieSul_L,SylParieSul_L,640
7,CSF,Sulcus_L,SylvianFissureExt_L,SylTempSul_L,SylTempSul_L,1598
8,CSF,Sulcus_L,TempSul_L,TempSul_L,TempSul_L,1139
9,CSF,Sulcus_R,CentralSul_R,CentralSul_R,CentralSul_R,1750


In [15]:
# Combine all hierarchical connections

links = []

levels = ["icv", "level1", "level2", "level3", "level4", "roi"]

for i in range(len(levels) - 1):

    # Group by the full hierarchy
    grouped = (
        subjectData.groupby(levels[:i + 2])["volume"]
        .sum()
        .reset_index()
    )

    for _, row in grouped.iterrows():

        # Include the full path to distinguish repeated region names
        source = "|".join(str(row[col]) for col in levels[:i + 1])
        target = "|".join(str(row[col]) for col in levels[:i + 2])

        links.append({
            "source": source,
            "target": target,
            "value": row["volume"]
        })

# Convert connections into a DataFrame
links_df = pd.DataFrame(links)

# View the first 10 connections
links_df.head(10)

,source,target,value
0,ICV,ICV|CSF,109787
1,ICV,ICV|Diencephalon_L,11817
2,ICV,ICV|Diencephalon_R,11560
3,ICV,ICV|Mesencephalon,10271
4,ICV,ICV|Metencephalon,159411
5,ICV,ICV|Myelencephalon,4975
6,ICV,ICV|Telencephalon_L,531153
7,ICV,ICV|Telencephalon_R,543444
8,ICV|CSF,ICV|CSF|Sulcus_L,33975
9,ICV|CSF,ICV|CSF|Sulcus_R,30018


In [18]:
# Create a list of unique nodes
nodes = pd.unique(
    pd.concat([links_df["source"], links_df["target"]])
)

# Assign a numerical ID to each node
node_ids = {node: i for i, node in enumerate(nodes)}

# Convert sources and targets into numerical IDs
links_df["source_id"] = links_df["source"].map(node_ids)
links_df["target_id"] = links_df["target"].map(node_ids)

# Check the results
links_df.head(10)

,source,target,value,source_id,target_id
0,ICV,ICV|CSF,109787,0,1
1,ICV,ICV|Diencephalon_L,11817,0,2
2,ICV,ICV|Diencephalon_R,11560,0,3
3,ICV,ICV|Mesencephalon,10271,0,4
4,ICV,ICV|Metencephalon,159411,0,5
5,ICV,ICV|Myelencephalon,4975,0,6
6,ICV,ICV|Telencephalon_L,531153,0,7
7,ICV,ICV|Telencephalon_R,543444,0,8
8,ICV|CSF,ICV|CSF|Sulcus_L,33975,1,9
9,ICV|CSF,ICV|CSF|Sulcus_R,30018,1,10


In [20]:
# Create labels for the nodes
node_labels = [node.split("|")[-1] for node in nodes]

# Create the Sankey diagram
sankey_fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=15,
                thickness=15,
                label=node_labels
            ),
            link=dict(
                source=links_df["source_id"],
                target=links_df["target_id"],
                value=links_df["value"]
            )
        )
    ]
)

# Adjust the layout
sankey_fig.update_layout(
    title_text="MRICloud Brain Volume Hierarchy - Subject 127 (Type 1)",
    font_size=8,
    height=1600,
    width=1800,
    margin=dict(l=30, r=30, t=80, b=30)
)

# Display the interactive diagram
sankey_fig.show()

In [23]:
# Save the interactive Sankey diagram as an HTML file

sankey_fig.write_html("sankey.html")

Part 3: GitHub

## Part 3: Publicly Hosted Sankey Diagram

The interactive MRICloud Sankey diagram for Subject 127 (Type 1) is publicly hosted using GitHub Pages.

[View the Interactive Sankey Diagram](https://amandaashleykim.github.io/hw5-sankey/sankey.html)